# 🎧 OmniVoice-Thai อย่างเดียว — ฟังเร็ว ๆ ใน ~8 นาที

ทดสอบเฉพาะ [hotdogs/omnivoice-thai](https://huggingface.co/hotdogs/omnivoice-thai) (OmniVoice ไฟน์จูนไทย, ไลเซนส์ฐาน Apache-2.0)
โคลนเสียงจากตัวอย่างเสียงเปรมวดี แล้วพูดประโยคทดสอบ 6 แบบ

**วิธีใช้:** Runtime → Change runtime type → **T4 GPU** → Save แล้ว Runtime → **Run all** → ฟังที่เซลล์สุดท้าย

In [ ]:
# 1) ติดตั้ง (~2 นาที)
!pip install -q omnivoice edge-tts soundfile
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ไม่มี — แนะนำเปิด T4 ไม่งั้นช้ามาก')

In [ ]:
# 2) ประโยคทดสอบ + เสียงอ้างอิงสำหรับโคลน
#    (อยากใช้เสียงตัวเอง: อัปโหลด .wav ~10 วิ แล้วเปลี่ยน REF_AUDIO / REF_TEXT)
import os, time
os.makedirs('samples_omni', exist_ok=True)

SENTENCES = {
  '01_basic': 'สวัสดีครับ ยินดีต้อนรับเข้าสู่ช่องของเรา วันนี้มีเรื่องราวน่าสนใจมาเล่าให้ฟังกันครับ',
  '02_story': 'ลองจินตนาการดูนะครับ ชายคนหนึ่งตัดสินใจเดินออกจากช่องยูทูบที่มีผู้ติดตามยี่สิบล้านคน โดยไม่บอกใครแม้แต่คำเดียว',
  '03_numbers_raw': 'เมื่อเวลา 18:30 น. ของวันที่ 2 สิงหาคม 2569 ยอดวิวพุ่งไปถึง 2,500,000 วิว สร้างรายได้กว่า 64,000 บาท',
  '04_english_mix': 'ช่อง YouTube ของเราใช้ AI ทำคลิป Shorts ลง TikTok และ Facebook ทุกวัน',
  '05_english_thai_spelled': 'ช่องยูทูบของเราใช้เอไอทำคลิปช็อตส์ ลงติ๊กต่อกและเฟซบุ๊กทุกวัน',
  '06_emotion': 'อะไรนะครับ! เขาทิ้งรายได้เดือนละหลายล้านบาทไปเฉย ๆ อย่างนั้นเหรอ เรื่องนี้มันบ้ามาก ๆ ไม่มีใครเชื่อเลยจริง ๆ',
}
timing = {}

import edge_tts
await edge_tts.Communicate(SENTENCES['01_basic'], 'th-TH-PremwadeeNeural').save('ref_premwadee.mp3')
!ffmpeg -y -loglevel error -i ref_premwadee.mp3 -ar 24000 -ac 1 ref_premwadee.wav
REF_AUDIO = 'ref_premwadee.wav'
REF_TEXT = SENTENCES['01_basic']
print('พร้อม: เสียงอ้างอิง +', len(SENTENCES), 'ประโยค')

In [ ]:
# 3) OmniVoice-Thai — โหลดโมเดลแล้วเจนครบ 6 ประโยค (โหลดครั้งแรก ~3-4 นาที)
assert 'SENTENCES' in globals(), '⚠️ ยังไม่ได้รันเซลล์ 2 — กด ▶ เซลล์ 2 ก่อน'
import soundfile as sf
from omnivoice import OmniVoice

dev = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dt = torch.float16 if torch.cuda.is_available() else torch.float32
MODEL_ID = 'hotdogs/omnivoice-thai'  # ถ้าโหลดไม่ได้ ถอยไปตัวฐาน k2-fsa/OmniVoice
try:
    model = OmniVoice.from_pretrained(MODEL_ID, device_map=dev, dtype=dt)
except Exception as e:
    print('โหลดตัวไฟน์จูนไทยไม่ได้ ->', e, '\nใช้ตัวฐานแทน')
    MODEL_ID = 'k2-fsa/OmniVoice'
    model = OmniVoice.from_pretrained(MODEL_ID, device_map=dev, dtype=dt)
print('ใช้โมเดล:', MODEL_ID)

for sid, text in SENTENCES.items():
    try:
        t0 = time.time()
        audio = model.generate(text=text, ref_audio=REF_AUDIO, ref_text=REF_TEXT)
        sf.write(f'samples_omni/omni_thai_{sid}.wav', audio[0], 24000)
        timing[f'omni_thai_{sid}'] = time.time() - t0
        print(f'เสร็จ: {sid} ({timing[f"omni_thai_{sid}"]:.1f} วิ)')
    except Exception as e:
        print('ข้าม', sid, '->', e)

In [ ]:
# 4) 🎧 ฟังทั้ง 6 ประโยค (มีเสียงอ้างอิงเปรมวดีให้เทียบบนสุด)
import glob, os
from IPython.display import Audio, display, Markdown

display(Markdown('### เสียงอ้างอิงที่ใช้โคลน (เปรมวดี)'))
display(Audio('ref_premwadee.wav'))

for sid in SENTENCES:
    files = [f for f in sorted(glob.glob(f'samples_omni/*_{sid}.*')) if os.path.getsize(f) > 0]
    if not files:
        continue
    display(Markdown(f'---\n### {sid}\n> {SENTENCES[sid]}'))
    for f in files:
        name = f.split('/')[-1]
        secs = timing.get(name.rsplit('.', 1)[0])
        display(Markdown(f'**{name}**' + (f' — {secs:.1f} วิ' if secs else '')))
        display(Audio(f))

In [ ]:
# 5) (ทางเลือก) โหลดเก็บเป็น zip
!zip -q -r omni_thai_samples.zip samples_omni
try:
    from google.colab import files
    files.download('omni_thai_samples.zip')
except Exception as e:
    print('ดาวน์โหลดอัตโนมัติไม่ได้:', e)
    print('ดึงเองได้ที่แถบ Files (ไอคอนโฟลเดอร์ 📁 ซ้ายมือ)')